In [66]:
# show CCC good for 5k
from simulate_data import *
from scone_tools.algorithms.SCoNE import SCoNE_parallel
from scone_tools.evaluation.reconstruction_evaluation import calculate_ccc
import pickle
root_dir = '/home/jupyter/repos/SCoNE'
code_dir = f'{root_dir}/code'
tmp_folder = f"{root_dir}/output/tmp"
experiment_name = 'CCC_test'
exp_tmp_folder = f'{tmp_folder}/{experiment_name}'
exp_models_folder = f'{output_dir}/models/{experiment_name}'
os.makedirs(exp_tmp_folder, exist_ok=True)
os.makedirs(exp_models_folder, exist_ok=True)

DEFAULT_SIM_KWARGS = {"n":50000,"M_C":1000,"M_G":1000,"rank":3,"noise":0.5,"gamma": 5,"sparsity":0.2,"rG":0.5,
                  "rGC":0.2,"rZ":0.2,"signed_cov_effects": False,"subgroup_structure": True, "overdispersion_nu":0} 
sim = simulate_views(**DEFAULT_SIM_KWARGS)
num_init=10
factor_matrices, loss_function, benchmark_info = SCoNE_parallel(
    sim["G"].astype(float), 
    sim["C"].astype(float), 
    sim["Z"].astype(float), 
    rank=3,
    alpha=0,
    lambda_H_G=0, 
    lambda_H_C=0, 
    lambda_Gloss=1,  
    num_init=num_init,
    init='random',
    G_loss_type='kl_div', 
    C_loss_type='kl_div', 
    write_all_init=True,
    write_all_init_path=f'{exp_models_folder}/run'
)


KeyboardInterrupt: 

In [65]:
sim['G'].shape

(1200, 20)

In [60]:
W_list = []
for run in range(50):
    with open(f'{exp_models_folder}/run_{run}_factor_matrices.pkl', 'rb') as f:
        factor_matrices = pickle.load(f)
    W_list.append(factor_matrices["W"])

results = [] 
for N in [5000,10000,15000,25000,50000]:
    prev_idx = set()
    for seed in range(10):
        coph_corr, idx = calculate_ccc(W_list, max_samples=N, random_state=seed)
        assert set(idx) != prev_idx
        prev_idx = set(idx)
        results.append([N, seed, coph_corr])

results = pd.DataFrame(results, columns=["N", "seed", "coph_corr"])

AssertionError: 

In [62]:
len(set(idx))

1200